# 14 · Temporal Fusion Transformer

Task 16.7b. A Temporal Fusion Transformer (Lim et al., 2021) shared by all stocks, written in PyTorch
(`src/models/tft.py`) so it shares the LSTM's data, walk-forward runner and forecast table. Inputs: symbol and sector
(static), the 34 stock, market and macro features over a 30-day window (observed), and the 5 calendar features for the window and the next 10
trading days (known in advance; the catalyst calendar, task 7.x, will add results and RBI dates here). It forecasts
quantiles of the cumulative return to each of the next 10 days; `p_up`, the expected return and volatility come from
those quantiles. Its forecasts are also the base forecast for the Forecaster agent (task 12.6).

CPU settings (`models.tft.sequence` in `config.yaml`): 30-day window, 30,000 random samples per epoch, up to 15 epochs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.analysis.plots import (BLUE_CMAP, HORIZON_COLORS, MODEL_COLORS, TEXT_2, apply_style, bar_style,
                                dot_compare, figures_dir, save)
from src.config import Config
from src.evaluation.metrics import calibration_table, evaluate, join_labels
from src.models.data import label_frame, load_panel
from src.models.walkforward import run_or_load

cfg = Config.load()
apply_style()
pd.set_option("display.width", 160)
pd.set_option("display.precision", 4)
val_start, val_end = cfg.period("validation")

In [ ]:
RETRAIN = False          # True retrains even when saved forecasts match the current settings
panel = load_panel(cfg)
labels = label_frame(panel)
fc = pd.concat([run_or_load(m, cfg, "validation", panel=panel, retrain=RETRAIN)
                for m in ["tft", "base_rate", "hist_vol"]], ignore_index=True)
metrics = evaluate(fc, labels)
metrics[metrics.model == "tft"].to_csv(figures_dir(cfg) / "tft_validation.csv", index=False)
metrics[["model", "horizon_days", "n", "accuracy", "f1_up", "brier", "log_loss", "predicted_up", "vol_mae", "vol_rmse"]]

## Training curve (fold trained to the end of 2022)

In [ ]:
import json
fold = sorted((cfg.models_dir / "tft" / "validation").glob("fold_*"))[0]
history = pd.DataFrame(json.loads((fold / "history.json").read_text()))
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(history.epoch, history.train_loss, marker="o", color=HORIZON_COLORS[1], label="training")
ax.plot(history.epoch, history.val_loss, marker="o", color=HORIZON_COLORS[10], label="early-stopping hold-out")
best = history.val_loss.idxmin()
ax.annotate("kept", (history.epoch[best], history.val_loss[best]), xytext=(0, -16), textcoords="offset points",
            ha="center", color=TEXT_2, fontsize=9)
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.set_title("TFT training loss")
ax.legend()
save(fig, cfg, "14_tft_training_curve")
plt.show()
history.round(4)

## Against the baselines

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
dot_compare(axes[0], metrics, "accuracy", ["tft", "base_rate"], panel.horizons, reference=0.5)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].set_title("Direction accuracy")
dot_compare(axes[1], metrics, "brier", ["tft", "base_rate"], panel.horizons, fmt="{:.4f}")
axes[1].set_title("Brier score (lower is better)")
dot_compare(axes[2], metrics, "vol_mae", ["tft", "hist_vol"], panel.horizons, fmt="{:.4f}")
axes[2].set_title("Volatility MAE (lower is better)")
fig.suptitle("TFT vs baselines, validation 2023", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, cfg, "14_tft_vs_baselines")
plt.show()

## Calibration: forecast probability vs how often the stock rose

In [ ]:
cal = calibration_table(join_labels(fc[fc.model == "tft"], labels), bins=10)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
for h in panel.horizons:
    g = cal[cal.horizon_days == h]
    ax.plot(g.p_mean, g.up_rate, marker="o", color=HORIZON_COLORS[h], label=f"{h}-day")
lims = [min(cal.p_mean.min(), cal.up_rate.min()) - 0.02, max(cal.p_mean.max(), cal.up_rate.max()) + 0.02]
ax.plot(lims, lims, color=TEXT_2, linewidth=1, linestyle="--")
ax.set_xlabel("mean forecast p_up (decile)")
ax.set_ylabel("actual share of up moves")
ax.set_title("TFT calibration, validation 2023")
ax.legend()
save(fig, cfg, "14_tft_calibration")
plt.show()
cal.pivot(index="bin", columns="horizon_days", values=["p_mean", "up_rate"]).round(3)

## Which inputs the model relies on (variable-selection weights, validation rows)

In [ ]:
from src.models.tft import TFTModel

model = TFTModel.load(cfg, fold, panel)
mask = panel.eval_mask(val_start, val_end)
importance = model.variable_importance(mask)
past = importance[importance.part == "past"].head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4.6))
ax.barh(past.input, past.weight, color=MODEL_COLORS["tft"], height=0.6, **bar_style())
ax.set_xlabel("mean selection weight (inputs sum to 1)")
ax.set_title("TFT: most-used inputs over the 30-day window")
ax.grid(axis="y", visible=False)
save(fig, cfg, "14_tft_variable_importance")
plt.show()
importance.round(4)

## In-sample vs out-of-sample

How well the median forecast ranks 5-day returns (Spearman) on a slice of training rows, on the early-stopping
hold-out and in validation. A model that ranks well only in-sample has learned time-specific patterns.

In [ ]:
from scipy.stats import spearmanr

checks = []
for name, a, b in [("train (H1 2019)", "2019-01-01", "2019-06-30"), ("hold-out (H2 2022)", "2022-06-15", "2022-12-16"),
                   ("validation 2023", val_start, val_end)]:
    m = panel.df.ready.to_numpy() & (panel.df.trade_date >= pd.Timestamp(a)).to_numpy() & (panel.df.trade_date <= pd.Timestamp(b)).to_numpy()
    r, qq = model.predict_quantiles(panel, m)
    y = panel.df["fwd_ret_5"].to_numpy()[r]
    ok = ~np.isnan(y)
    med = qq[ok, 4, 2]
    checks.append({"rows": name, "spearman_median_vs_actual": spearmanr(med, y[ok]).statistic,
                   "median_above_0": float((med > 0).mean()), "actual_up": float((y[ok] > 0).mean()),
                   "mean_median": float(med.mean()), "mean_actual": float(y[ok].mean())})
pd.DataFrame(checks)

## Do the 10%–90% return intervals hold 80% of outcomes?

In [ ]:
rows, q = model.predict_quantiles(panel, mask)
coverage = []
for h in panel.horizons:
    actual = panel.df[f"fwd_ret_{h}"].to_numpy()[rows]
    ok = ~np.isnan(actual)
    lo, hi = q[ok, h - 1, 0], q[ok, h - 1, -1]
    coverage.append({"horizon_days": h, "inside_10_90": float(((actual[ok] >= lo) & (actual[ok] <= hi)).mean()),
                     "below_10": float((actual[ok] < lo).mean()), "above_90": float((actual[ok] > hi).mean()),
                     "median_width": float(np.median(hi - lo))})
pd.DataFrame(coverage)

## Findings

1. **Training:** 30,000 samples per epoch at about 90 seconds each. Early stopping kept epoch 1 of 4: the hold-out loss rose after every later epoch (1.039 to 1.417), so with these inputs the model overfits as soon as it trains further.
2. **Direction:** below the base rate: 51.4% / 45.7% / 42.6% vs 53.1% / 56.2% / 60.6%. It predicts a 5-day fall for 89% of stock-days in a year when 56% rose. Brier 0.2510 / 0.2539 / 0.2552 vs 0.2492 / 0.2485 / 0.2441.
3. **Its probabilities rank in the right order but sit too low.** At 5 days the top decile of p_up rose 59.1% of the time and the bottom decile 54.3% (10 days: 66.2% vs 57.4%), yet the mean median forecast is -0.35% against an actual +0.44%. The ranking comes from timing the market (pooled Spearman with the 5-day return +0.02), not from picking stocks (within-day Spearman -0.015).
4. **Overfitting check:** the median forecast's Spearman with the 5-day return is 0.070 on 2019 training rows, 0.048 on the late-2022 hold-out and 0.025 in 2023.
5. **Intervals and volatility:** the 10%-90% intervals hold 98% / 95% / 91% of outcomes instead of 80%, and volatility MAE is 0.0230 / 0.0125 / 0.0072 vs historical volatility's 0.0088 / 0.0057 / 0.0050: the one-epoch model has not learned the spread.
6. **Inputs used:** over the forecast days the holiday gaps dominate (next_gap_days 0.60, gap_days 0.23); over the past window relative strength vs the Nifty (0.136), distance from EMA 50 (0.133), the 20-day return (0.081), the US VIX level (0.076) and the overnight gap (0.074) lead.
7. **Next (tune on validation only):** calibrate the probability level on validation (task 15.2), since the ranking is right but the level is not; stronger regularisation (dropout 0.3, hidden 16) with a lower learning rate (3e-4); drop level inputs that act as date stamps (us10y, usvix); add results and RBI dates as known-future inputs once the catalyst calendar (7.x) exists.